# Sample comparison notebook

This notebook is the analysis workspace for comparing samples.

In [ ]:
from pathlib import Path
import pandas as pd

# Configure these
results_root = Path("/home/peterkad/pkadmaster/indel_scanner/results")
samples = ["tr_plus_unmapped_diploid_v2", "ph_plus_unmapped_diploid_v2"]


def latest_run_dir(sample_root: Path) -> Path | None:
    if not sample_root.exists():
        return None
    run_dirs = [p for p in sample_root.iterdir() if p.is_dir()]
    if not run_dirs:
        return None
    return max(run_dirs, key=lambda p: p.stat().st_mtime)


records = []
missing = []

for sample in samples:
    sample_root = results_root / sample
    run_dir = latest_run_dir(sample_root)
    if run_dir is None:
        missing.append((sample, "no_run_dir"))
        continue
    per_type = run_dir / "per_type_mutation_frequency.tsv"
    callable_bases = run_dir / "callable_bases.tsv"
    if not per_type.exists() or not callable_bases.exists():
        missing.append((sample, str(run_dir)))
        continue
    records.append(
        {
            "sample": sample,
            "run_dir": run_dir,
            "per_type": per_type,
            "callable_bases": callable_bases,
        }
    )

pd.DataFrame(records), pd.DataFrame(missing, columns=["sample", "issue"])

In [ ]:
def load_per_type(path: Path, sample: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t")
    df["sample"] = sample
    drop_cols = ["unique_rate", "unique_sites", "str_region_count"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    return df


def load_callable(path: Path, sample: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t")
    df["sample"] = sample
    drop_cols = ["str_region_count"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    return df


per_type_dfs = []
callable_dfs = []

for rec in records:
    sample = rec["sample"]
    per_type_dfs.append(load_per_type(rec["per_type"], sample))
    callable_dfs.append(load_callable(rec["callable_bases"], sample))

per_type_all = pd.concat(per_type_dfs, ignore_index=True) if per_type_dfs else pd.DataFrame()
callable_all = pd.concat(callable_dfs, ignore_index=True) if callable_dfs else pd.DataFrame()

per_type_all.head()

In [ ]:
# Compare frequency by sample and STR class
comparison = (
    per_type_all.groupby(["sample", "str_class"], as_index=False)[["count", "callable_bases"]]
    .sum()
)
comparison["frequency"] = comparison["count"] / comparison["callable_bases"]

comparison = comparison.sort_values(["str_class", "sample"])
comparison.style.format({"frequency": "{:.2e}", "callable_bases": "{:.0f}"})